In [2]:
# ============================================================
# VQA 5개 모델 Soft-voting 앙상블
#
# Google Drive 구조:
#
# MyDrive/
# └── ensemble/
#     ├── qwen35_27b_test_probabilities_03_final_other.csv
#     ├── qwen38_27b_test_probabilities_03_final_other.csv
#     ├── qwen35_4b_test_probabilities_03_final_other.csv
#     ├── gemma_e2b_test_probabilities_03_final_other.csv
#     └── gemma_31b_test_probabilities_03_final_other.csv
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Google Drive 마운트
# ============================================================

DEFAULT_DRIVE_ROOT = Path("/content/drive/MyDrive")

if DEFAULT_DRIVE_ROOT.is_dir():
    DRIVE_ROOT = DEFAULT_DRIVE_ROOT
    print(f"기존 Google Drive 사용: {DRIVE_ROOT}")

else:
    from google.colab import drive

    # /content/drive에 기존 파일이 있을 경우를 대비해
    # 별도의 빈 위치에 마운트
    mount_index = 0

    while True:
        if mount_index == 0:
            mount_point = Path("/content/gdrive_mount")
        else:
            mount_point = Path(
                f"/content/gdrive_mount_{mount_index}"
            )

        if not mount_point.exists():
            mount_point.mkdir(parents=True)
            break

        if mount_point.is_dir() and not any(mount_point.iterdir()):
            break

        mount_index += 1

    drive.mount(
        str(mount_point),
        force_remount=False,
    )

    DRIVE_ROOT = mount_point / "MyDrive"

print(f"DRIVE_ROOT: {DRIVE_ROOT}")


# ============================================================
# 2. 입력·출력 경로
# ============================================================

ENSEMBLE_ROOT = DRIVE_ROOT / "Ensemble"
OUTPUT_ROOT = ENSEMBLE_ROOT / "outputs"

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_FILES = {
    "qwen38_27b": (
        ENSEMBLE_ROOT
        / "qwen38_27b_test_probabilities_03_final_other.csv"
    ),

    "qwen35_27b": (
        ENSEMBLE_ROOT
        / "qwen35_27b_test_probabilities_03_final_other.csv"
    ),

    "qwen35_4b": (
        ENSEMBLE_ROOT
        / "qwen35_4b_test_probabilities_03_final_other.csv"
    ),

    "gemma4_31b": (
        ENSEMBLE_ROOT
        / "gemma_31b_test_probabilities_03_final_other.csv"
    ),

    "gemma4_e2b": (
        ENSEMBLE_ROOT
        / "gemma_e2b_test_probabilities_03_final_other.csv"
    ),
}

LABELS = np.array(["a", "b", "c", "d"])
PROBABILITY_COLUMNS = ["p_a", "p_b", "p_c", "p_d"]

# 대회 테스트 데이터의 예상 행 수
EXPECTED_ROW_COUNT = 6714

print(f"ENSEMBLE_ROOT: {ENSEMBLE_ROOT}")
print(f"OUTPUT_ROOT  : {OUTPUT_ROOT}")


# ============================================================
# 3. 입력 파일 존재 검사
# ============================================================

print()
print("=" * 80)
print("입력 파일 확인")
print("=" * 80)

missing_files = {}

for model_name, path in MODEL_FILES.items():
    if path.is_file():
        print(f"[FOUND]   {model_name}")
        print(f"          {path}")
    else:
        print(f"[MISSING] {model_name}")
        print(f"          {path}")

        missing_files[model_name] = path

if missing_files:
    missing_message = "\n".join(
        f"- {model_name}: {path}"
        for model_name, path in missing_files.items()
    )

    raise FileNotFoundError(
        "\n다음 확률 CSV를 찾을 수 없습니다.\n"
        f"{missing_message}\n\n"
        "파일명과 MyDrive/ensemble 폴더 위치를 확인하세요."
    )


# ============================================================
# 4. 확률 CSV 로딩 및 검증
#
# 지원하는 확률 열:
#   p_a, p_b, p_c, p_d
#   a, b, c, d
#   prob_a, prob_b, prob_c, prob_d
#   score_a, score_b, score_c, score_d
# ============================================================

PROBABILITY_COLUMN_SCHEMES = [
    ["p_a", "p_b", "p_c", "p_d"],
    ["a", "b", "c", "d"],
    ["prob_a", "prob_b", "prob_c", "prob_d"],
    ["score_a", "score_b", "score_c", "score_d"],
]


def load_probability_csv(model_name, path):
    frame = pd.read_csv(path)

    frame.columns = [
        str(column).strip()
        for column in frame.columns
    ]

    # ID 열 검사
    if "id" not in frame.columns:
        raise ValueError(
            f"[{model_name}] id 열이 없습니다.\n"
            f"파일: {path}\n"
            f"현재 열: {frame.columns.tolist()}"
        )

    # 확률 열 형식 감지
    detected_columns = None

    for scheme in PROBABILITY_COLUMN_SCHEMES:
        if all(column in frame.columns for column in scheme):
            detected_columns = scheme
            break

    if detected_columns is None:
        raise ValueError(
            f"[{model_name}] a/b/c/d 확률 열을 찾지 못했습니다.\n"
            f"파일: {path}\n"
            f"현재 열: {frame.columns.tolist()}"
        )

    # 확률 열 이름 통일
    rename_map = dict(
        zip(
            detected_columns,
            PROBABILITY_COLUMNS,
        )
    )

    frame = frame.rename(columns=rename_map)

    # ID 문자열 표준화
    frame["id"] = (
        frame["id"]
        .astype(str)
        .str.strip()
    )

    if frame["id"].eq("").any():
        raise ValueError(
            f"[{model_name}] 빈 ID가 있습니다."
        )

    # 중복 ID 검사
    duplicate_mask = frame["id"].duplicated(
        keep=False
    )

    if duplicate_mask.any():
        duplicate_ids = (
            frame.loc[duplicate_mask, "id"]
            .unique()
            .tolist()
        )

        raise ValueError(
            f"[{model_name}] 중복 ID가 있습니다.\n"
            f"예시: {duplicate_ids[:10]}"
        )

    # 행 수 검사
    if len(frame) != EXPECTED_ROW_COUNT:
        raise ValueError(
            f"[{model_name}] 행 수가 다릅니다.\n"
            f"예상: {EXPECTED_ROW_COUNT:,}\n"
            f"실제: {len(frame):,}\n"
            "partial 파일이 아닌지 확인하세요."
        )

    # 확률을 숫자로 변환
    for column in PROBABILITY_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    probabilities = frame[
        PROBABILITY_COLUMNS
    ].to_numpy(dtype=np.float64)

    # NaN·무한대 검사
    finite_mask = np.isfinite(
        probabilities
    ).all(axis=1)

    if not finite_mask.all():
        invalid_rows = np.where(
            ~finite_mask
        )[0]

        raise ValueError(
            f"[{model_name}] NaN 또는 무한대가 있습니다.\n"
            f"문제 행 예시: {invalid_rows[:10].tolist()}"
        )

    # 음수 확률 검사
    if (probabilities < -1e-8).any():
        invalid_rows = np.where(
            (probabilities < -1e-8).any(axis=1)
        )[0]

        raise ValueError(
            f"[{model_name}] 음수 확률이 있습니다.\n"
            f"문제 행 예시: {invalid_rows[:10].tolist()}"
        )

    # 부동소수점 오차로 발생한 극소 음수 제거
    probabilities = np.clip(
        probabilities,
        0.0,
        None,
    )

    probability_sums = probabilities.sum(
        axis=1
    )

    if (probability_sums <= 0).any():
        invalid_rows = np.where(
            probability_sums <= 0
        )[0]

        raise ValueError(
            f"[{model_name}] 확률 합이 0인 행이 있습니다.\n"
            f"문제 행 예시: {invalid_rows[:10].tolist()}"
        )

    # 모든 모델의 확률 합을 정확히 1로 정규화
    probabilities = (
        probabilities
        / probability_sums[:, None]
    )

    normalized_frame = pd.DataFrame({
        "id": frame["id"].to_numpy(),
        "p_a": probabilities[:, 0],
        "p_b": probabilities[:, 1],
        "p_c": probabilities[:, 2],
        "p_d": probabilities[:, 3],
    })

    calculated_predictions = LABELS[
        np.argmax(
            probabilities,
            axis=1,
        )
    ]

    # 저장된 pred 또는 answer가 있다면 비교만 수행
    saved_prediction_column = None

    if "pred" in frame.columns:
        saved_prediction_column = "pred"
    elif "answer" in frame.columns:
        saved_prediction_column = "answer"

    if saved_prediction_column is not None:
        saved_predictions = (
            frame[saved_prediction_column]
            .astype(str)
            .str.strip()
            .str.lower()
            .to_numpy()
        )

        mismatch_count = int(
            np.sum(
                saved_predictions
                != calculated_predictions
            )
        )

        if mismatch_count > 0:
            print(
                f"[참고] {model_name}: "
                f"저장된 {saved_prediction_column}와 "
                f"확률 argmax가 {mismatch_count:,}개 다릅니다."
            )

    print(
        f"[LOADED] {model_name:<12} "
        f"rows={len(normalized_frame):,}, "
        f"columns={detected_columns}"
    )

    return normalized_frame


# ============================================================
# 5. 모든 모델 확률 로딩
# ============================================================

print()
print("=" * 80)
print("확률 CSV 검증")
print("=" * 80)

model_frames = {}

for model_name, path in MODEL_FILES.items():
    model_frames[model_name] = load_probability_csv(
        model_name=model_name,
        path=path,
    )


# ============================================================
# 6. 모델 간 ID 일치 검사 및 순서 정렬
# ============================================================

REFERENCE_MODEL = "qwen38_27b"

reference_frame = model_frames[REFERENCE_MODEL]
reference_ids = reference_frame["id"].tolist()
reference_id_set = set(reference_ids)
row_count = len(reference_ids)

aligned_probabilities = {}

for model_name, frame in model_frames.items():
    current_id_set = set(frame["id"])

    missing_ids = (
        reference_id_set
        - current_id_set
    )

    extra_ids = (
        current_id_set
        - reference_id_set
    )

    if missing_ids or extra_ids:
        raise ValueError(
            f"[{model_name}] ID 구성이 기준 모델과 다릅니다.\n"
            f"누락 ID: {len(missing_ids):,}개\n"
            f"추가 ID: {len(extra_ids):,}개\n"
            f"누락 예시: {sorted(missing_ids)[:10]}\n"
            f"추가 예시: {sorted(extra_ids)[:10]}"
        )

    aligned_frame = (
        frame
        .set_index("id")
        .loc[reference_ids]
    )

    aligned_probabilities[model_name] = (
        aligned_frame[PROBABILITY_COLUMNS]
        .to_numpy(dtype=np.float64)
    )

print()
print(
    f"ID 검증 완료: 모든 모델이 "
    f"{row_count:,}개의 동일한 ID를 가지고 있습니다."
)


# ============================================================
# 7. 앙상블 가중치
#
# 결과가 두 종류 생성됩니다.
#
# top3:
#   대형 모델 3개 중심
#
# five_model_family_balanced:
#   소형 모델까지 포함하고 Qwen/Gemma 계열을 분산
# ============================================================

WEIGHT_SETS = {
    "top3": {
        "qwen38_27b": 0.40,
        "gemma4_31b": 0.35,
        "qwen35_27b": 0.25,
    },

    "five_model_family_balanced": {
        "qwen38_27b": 0.30,
        "qwen35_27b": 0.20,
        "qwen35_4b": 0.05,
        "gemma4_31b": 0.38,
        "gemma4_e2b": 0.07,
    },
}


# ============================================================
# 8. Soft-voting 앙상블 함수
# ============================================================

def create_ensemble(
    ensemble_name,
    weights,
):
    total_weight = float(
        sum(weights.values())
    )

    if total_weight <= 0:
        raise ValueError(
            f"[{ensemble_name}] 가중치 합이 0 이하입니다."
        )

    normalized_weights = {
        model_name: weight / total_weight
        for model_name, weight in weights.items()
    }

    ensemble_probabilities = np.zeros(
        (row_count, 4),
        dtype=np.float64,
    )

    for model_name, weight in normalized_weights.items():
        ensemble_probabilities += (
            weight
            * aligned_probabilities[model_name]
        )

    ensemble_probabilities /= (
        ensemble_probabilities.sum(
            axis=1,
            keepdims=True,
        )
    )

    ensemble_predictions = LABELS[
        np.argmax(
            ensemble_probabilities,
            axis=1,
        )
    ]

    # --------------------------------------------
    # Kaggle 제출 파일
    # --------------------------------------------

    submission = pd.DataFrame({
        "id": reference_ids,
        "answer": ensemble_predictions,
    })

    submission_path = (
        OUTPUT_ROOT
        / f"submission_{ensemble_name}.csv"
    )

    submission.to_csv(
        submission_path,
        index=False,
        encoding="utf-8",
    )

    # --------------------------------------------
    # 상세 진단 파일
    # --------------------------------------------

    sorted_probabilities = np.sort(
        ensemble_probabilities,
        axis=1,
    )

    margins = (
        sorted_probabilities[:, -1]
        - sorted_probabilities[:, -2]
    )

    entropies = -np.sum(
        ensemble_probabilities
        * np.log(
            np.clip(
                ensemble_probabilities,
                1e-12,
                1.0,
            )
        ),
        axis=1,
    )

    diagnostics = pd.DataFrame({
        "id": reference_ids,
        "answer": ensemble_predictions,
        "p_a": ensemble_probabilities[:, 0],
        "p_b": ensemble_probabilities[:, 1],
        "p_c": ensemble_probabilities[:, 2],
        "p_d": ensemble_probabilities[:, 3],
        "confidence": ensemble_probabilities.max(axis=1),
        "margin": margins,
        "entropy": entropies,
    })

    model_prediction_list = []

    for model_name in normalized_weights:
        model_predictions = LABELS[
            np.argmax(
                aligned_probabilities[model_name],
                axis=1,
            )
        ]

        diagnostics[
            f"pred_{model_name}"
        ] = model_predictions

        diagnostics[
            f"confidence_{model_name}"
        ] = aligned_probabilities[
            model_name
        ].max(axis=1)

        model_prediction_list.append(
            model_predictions
        )

    prediction_matrix = np.column_stack(
        model_prediction_list
    )

    diagnostics["agreement_count"] = np.sum(
        prediction_matrix
        == ensemble_predictions[:, None],
        axis=1,
    )

    diagnostics[
        "unique_prediction_count"
    ] = np.array([
        len(set(row))
        for row in prediction_matrix
    ])

    diagnostics_path = (
        OUTPUT_ROOT
        / f"diagnostics_{ensemble_name}.csv"
    )

    diagnostics.to_csv(
        diagnostics_path,
        index=False,
        encoding="utf-8",
    )

    print()
    print("-" * 80)
    print(f"앙상블 완료: {ensemble_name}")
    print("-" * 80)

    for model_name, weight in normalized_weights.items():
        print(
            f"{model_name:<12}: "
            f"{weight:.4f}"
        )

    print(f"제출 파일: {submission_path}")
    print(f"진단 파일: {diagnostics_path}")

    return {
        "submission": submission,
        "diagnostics": diagnostics,
        "probabilities": ensemble_probabilities,
        "submission_path": submission_path,
        "diagnostics_path": diagnostics_path,
    }


# ============================================================
# 9. 앙상블 실행
# ============================================================

ensemble_results = {}

for ensemble_name, weights in WEIGHT_SETS.items():
    ensemble_results[ensemble_name] = create_ensemble(
        ensemble_name=ensemble_name,
        weights=weights,
    )


# ============================================================
# 10. 두 앙상블 예측 비교
# ============================================================

top3_answers = ensemble_results[
    "top3"
]["submission"]["answer"].to_numpy()

five_model_answers = ensemble_results[
    "five_model_family_balanced"
]["submission"]["answer"].to_numpy()

comparison = pd.DataFrame({
    "id": reference_ids,
    "answer_top3": top3_answers,
    "answer_five_model": five_model_answers,
})

comparison["is_different"] = (
    comparison["answer_top3"]
    != comparison["answer_five_model"]
)

disagreements = comparison.loc[
    comparison["is_different"]
].copy()

comparison_path = (
    OUTPUT_ROOT
    / "top3_vs_five_model_disagreements.csv"
)

disagreements.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 11. 최종 제출 파일 검증
# ============================================================

for ensemble_name, result in ensemble_results.items():
    submission = result["submission"]

    if len(submission) != EXPECTED_ROW_COUNT:
        raise RuntimeError(
            f"[{ensemble_name}] 제출 파일의 행 수가 잘못되었습니다."
        )

    if submission["id"].duplicated().any():
        raise RuntimeError(
            f"[{ensemble_name}] 제출 파일에 중복 ID가 있습니다."
        )

    invalid_answers = (
        set(submission["answer"])
        - set(LABELS)
    )

    if invalid_answers:
        raise RuntimeError(
            f"[{ensemble_name}] 잘못된 answer가 있습니다: "
            f"{invalid_answers}"
        )


# ============================================================
# 12. 결과 출력
# ============================================================

print()
print("=" * 80)
print("전체 앙상블 완료")
print("=" * 80)

print(f"전체 샘플 수: {row_count:,}")
print(
    "Top3와 5개 모델의 예측이 다른 샘플 수: "
    f"{len(disagreements):,}"
)

print()
print("생성된 파일:")

for result in ensemble_results.values():
    print(f"- {result['submission_path']}")
    print(f"- {result['diagnostics_path']}")

print(f"- {comparison_path}")

print()
print("첫 번째 제출 권장 파일:")
print(
    OUTPUT_ROOT
    / "submission_five_model_family_balanced.csv"
)

print()
print("두 번째 비교 제출 파일:")
print(
    OUTPUT_ROOT
    / "submission_top3.csv"
)

print()
print("5개 모델 앙상블 미리보기:")

display(
    ensemble_results[
        "five_model_family_balanced"
    ]["submission"].head(10)
)

Mounted at /content/gdrive_mount_1
DRIVE_ROOT: /content/gdrive_mount_1/MyDrive
ENSEMBLE_ROOT: /content/gdrive_mount_1/MyDrive/Ensemble
OUTPUT_ROOT  : /content/gdrive_mount_1/MyDrive/Ensemble/outputs

입력 파일 확인
[FOUND]   qwen38_27b
          /content/gdrive_mount_1/MyDrive/Ensemble/qwen38_27b_test_probabilities_03_final_other.csv
[FOUND]   qwen35_27b
          /content/gdrive_mount_1/MyDrive/Ensemble/qwen35_27b_test_probabilities_03_final_other.csv
[FOUND]   qwen35_4b
          /content/gdrive_mount_1/MyDrive/Ensemble/qwen35_4b_test_probabilities_03_final_other.csv
[FOUND]   gemma4_31b
          /content/gdrive_mount_1/MyDrive/Ensemble/gemma_31b_test_probabilities_03_final_other.csv
[FOUND]   gemma4_e2b
          /content/gdrive_mount_1/MyDrive/Ensemble/gemma_e2b_test_probabilities_03_final_other.csv

확률 CSV 검증
[참고] qwen38_27b: 저장된 pred와 확률 argmax가 5개 다릅니다.
[LOADED] qwen38_27b   rows=6,714, columns=['a', 'b', 'c', 'd']
[LOADED] qwen35_27b   rows=6,714, columns=['p_a', 'p_b', 'p_c', 'p_d'

,id,answer
0,test_0001.jpg,b
1,test_0002.jpg,c
2,test_0003.jpg,d
3,test_0004.jpg,d
4,test_0005.jpg,c
5,test_0006.jpg,d
6,test_0007.jpg,b
7,test_0008.jpg,c
8,test_0009.jpg,c
9,test_0010.jpg,a
